In [ ]:
import pandas as pd
import csv
import os
import warnings

# Tắt cảnh báo không cần thiết
warnings.filterwarnings('ignore')

def clean_homedy_data(input_file='data_homedy_full.csv', output_file='clear_homedy.csv'):
    print(f"🚀 Bắt đầu xử lý file: {input_file}")
    
    # 1. Đọc dữ liệu thô
    if not os.path.exists(input_file):
        print(f"❌ Lỗi: Không tìm thấy file '{input_file}'. Hãy chạy scraper trước.")
        return

    try:
        df = pd.read_csv(input_file)
        print(f"   -> Kích thước dữ liệu gốc: {df.shape}")
        
        # 2. Xử lý và chuẩn hóa các cột
        # Chuyển đổi sang numeric (ép kiểu), biến lỗi thành NaN rồi fill 0
        df['price'] = pd.to_numeric(df['price'], errors='coerce').fillna(0)
        df['area'] = pd.to_numeric(df['area'], errors='coerce').fillna(0)
        
        # 3. Lọc bỏ dữ liệu rác
        # Logic: Tin không có giá hoặc diện tích = 0 thường là tin lỗi hoặc spam
        df_clean = df[(df['price'] > 0) & (df['area'] > 0)].copy()
        
        # 4. Tạo DataFrame chuẩn theo Schema chung (để khớp với Chotot)
        # Mapping các cột từ Homedy sang format chung
        new_df = pd.DataFrame({
            'url': df_clean['url'],
            'title': df_clean['title'],
            'price': df_clean['price'],
            'area': df_clean['area'],
            'district': df_clean['district'].fillna(''),
            # Tạo cột address dự phòng từ district để khớp schema Chotot nếu cần gộp
            'address': df_clean['district'].fillna(''), 
            'description': df_clean['description'].fillna(''),
            'source': 'homedy.com'
        })
        
        print(f"   -> Kích thước sau khi làm sạch: {new_df.shape}")
        print(f"   -> Đã loại bỏ {len(df) - len(new_df)} dòng dữ liệu lỗi (giá/diện tích = 0).")
        
        # 5. Xuất file CSV sạch
        if not new_df.empty:
            new_df.to_csv(
                output_file,
                index=False,
                quoting=csv.QUOTE_ALL,      # Bao quanh text bằng "" để tránh lỗi dấu phẩy trong nội dung
                escapechar='\\',            # Escape ký tự đặc biệt
                encoding='utf-8-sig'        # Encoding hỗ trợ tiếng Việt
            )
            print(f"✅ Thành công! File sạch đã được lưu tại: {output_file}")
        else:
            print("⚠️ Cảnh báo: Không có dữ liệu nào hợp lệ để xuất file.")

    except Exception as e:
        print(f"❌ Có lỗi xảy ra trong quá trình xử lý: {e}")

if __name__ == "__main__":
    clean_homedy_data()